# 2d Wave equation with hyperboloidal compactification in first order form with mass lumping and explicit time-stepping

In [1]:
from ngsolve import *
from netgen.occ import *
from ngsolve.webgui import Draw
from time import time


Rout = 2
domain = Circle((0, 0), Rout).Face()
#domain.edges.name = 'inner'
#domain.faces.name = 'inner'

domain.edges.name = 'outer'
domain.faces.name = 'outer'


RScat = 0.3


R=1.
inner = Circle((0, 0), R).Face()
inner.edges.name = 'interface'
inner.faces.name = 'inner'


if RScat == 0:
    Draw(Glue([domain-inner,inner]))
    geo = OCCGeometry(Glue([domain-inner,inner]), dim=2)

else:
    scatterer = Circle((0,0),RScat).Face()
    scatterer.edges.name = 'scatterer'


if RScat<R and RScat >0:
    Draw(Glue([domain-inner,inner-scatterer]))
    geo = OCCGeometry(Glue([domain-inner,inner-scatterer]), dim=2)
else:
    Draw(domain-scatterer)
    geo = OCCGeometry(domain-scatterer, dim=2)



WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'ngsolve_version': 'Netgen x.x', 'mesh_dim': 3…

In [2]:
mesh = Mesh(geo.GenerateMesh(maxh=0.1))
order = 2
mesh.Curve(order)
Draw(mesh)
print(mesh.GetMaterials())
print(mesh.GetBoundaries())

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.25…

('outer', 'inner')
('outer', 'interface', 'scatterer')


In [3]:

vx = CF((x,y))
rho = sqrt(x**2+y**2)
rho_ = rho-R

# using \omega = R/r^(1/2)

r = rho_/(1-rho_)+1
r_ = 1/(1-rho_)**2
H = 1-1/r_    #characteristic preserving




Pi = OuterProduct(vx,vx)/rho**2
Pi_perp = Id(2)-Pi

#a = R**2*(1-H**2)*r_/rho
#a_p = R**2*(1+H)/rho     #characteristic preserving
a_p = mesh.MaterialCF({'inner': 1, 'outer': R**2*(1+H)/rho})

#a_v = (r_*rho/r**2*Pi+1/rho/r_*Pi_perp)
a_v = mesh.MaterialCF({'inner': Id(2), 'outer': r_*rho/r**2*Pi+1/rho/r_*Pi_perp})

b = R**2*H/rho**2*vx

n = specialcf.normal(2)

In [4]:
# using dg and explicit time-stepping

fes_p = L2(mesh,order=order+1) 
#fes_p = H1(mesh,order=order)
fes_v = VectorL2(mesh,order=order)

fes_tr = FacetFESpace(mesh, order=order+1)
traceop = fes_p.TraceOperator(fes_tr, True)


p,p_ = fes_p.TnT()
v, v_ = fes_v.TnT()
phat,phat_ = fes_tr.TnT()


M_p_ = a_p*p*p_*dx
M_v_ = a_v*v*v_*dx

A_vp_ = (
    -v*grad(p_)*dx('inner')+p_*(v*n) * dx('inner',element_boundary=True)
    -1/r*v*grad(p_)*dx('outer')+1/r*p_*(v*n) * dx('outer',element_boundary=True)
    +r_/r**2/rho/2*InnerProduct(vx,v)*p_*dx('outer')
    )

A_vp_trace_ = -phat_ * (v*n) * dx('inner',element_boundary=True)-1/r * phat_ * (v*n) * dx('outer',element_boundary=True)

A_pp_ = (
    -InnerProduct(grad(p),b)*p_*dx('outer')
    -InnerProduct(b,n)*p_*p*ds('outer',skeleton=True)
    +InnerProduct(grad(p_),b)*p*dx('outer')
    )

A_pp_trace_ = (b*n) * phat_ * p * dx('outer',element_boundary=True)

A_pv_ = (
    grad(p)*v_*dx('inner')-p*(v_*n) * dx('inner',element_boundary=True)
    +1/r*v_*grad(p)*dx('outer')-1/r*p*(v_*n) * dx('outer',element_boundary=True)
    -r_/r**2/rho/2*InnerProduct(vx,v_)*p*dx('outer' )
    )

A_pv_trace_ = phat * (v_*n) * dx('inner',element_boundary=True)+1/r * phat * (v_*n) * dx('outer',element_boundary=True)


In [5]:
#M_p = BilinearForm(M_p_).Assemble().mat
#M_p = BilinearForm(M_p_).Assemble().mat
#M_p_inv = M_p.Inverse()
M_p_inv = fes_p.InvM(a_p)

In [6]:
#M_v = BilinearForm(M_v_).Assemble().mat
#M_v_inv = M_v.Inverse()
M_v_inv = fes_v.InvM(a_v)
#pl.spy(M_v.ToDense())

In [7]:
A_pp_el = BilinearForm(A_pp_).Assemble().mat
A_pp_tr = BilinearForm(A_pp_trace_).Assemble().mat
#A_vp_el = BilinearForm(A_vp_).Assemble().mat
A_pv_el = BilinearForm(A_pv_).Assemble().mat

#A_vp_tr = BilinearForm(A_vp_trace_).Assemble().mat
A_pv_tr = BilinearForm(A_pv_trace_).Assemble().mat


#A_vp = A_vp_el + traceop.T@A_vp_tr
A_pv = A_pv_el + A_pv_tr@traceop
A_vp = -A_pv.T

A_pp = A_pp_el + traceop.T@A_pp_tr-A_pp_tr.T@traceop

used dof inconsistency
(silence this warning by setting BilinearForm(...check_unused=False) )


In [8]:
v0 = CF((0,0))
p0 = exp(-30*((x-0.4)**2+(y-0.)**2))



dt = 1e-3

gfv = GridFunction(fes_v)
gfp = GridFunction(fes_p)
gfv.Set(v0)
gfp.Interpolate(p0)
scenep = Draw(gfp,mesh,min = 0, max = 0.1,order = order)

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.25…

In [9]:
tmpp = gfp.vec.CreateVector()

endt = 3
t = 0

drawevery = (.2/dt)
i = 0
comptime = 0
gfanim = GridFunction(fes_p,multidim = 0)

gfv.vec.data += 1/2*dt*M_v_inv@A_pv*gfp.vec
while t < endt:
    now = time()
    t += dt
    i += 1
    tmpp.data = A_vp*gfv.vec+A_pp*gfp.vec
    gfp.vec.data += dt*M_p_inv*tmpp
    gfv.vec.data += dt*M_v_inv@A_pv*gfp.vec
    comptime += time()-now
    if (i%drawevery) == 0:
        print('\r t = {}'.format(t),end="")
        scenep.Redraw()
        gfanim.AddMultiDimComponent(gfp.vec)
print("\n time per step: {}s".format(comptime/i))

 t = 2.59999999999982475

KeyboardInterrupt: 

In [ ]:
sceneanim = Draw(gfanim,animate=True,min=0,max=0.1)